# S - standard deviations for the reject, specification and reproduction tables


In [ ]:
# ============================================================================
#  SETUP - run this cell first. Change SMOKE and, if you like, N_JOBS.
# ============================================================================
SMOKE = False         # True: ~1 feeder, 1 seed, small N, minutes. False: full run.

# Optional: restrict this execution to a subset of work units. Leave empty for everything.
# Set e.g. ONLY_FEEDERS = [123] to run just the largest feeder, or ONLY_KS = [2, 3] to run a
# slice of the k grid. Every notebook is resumable, so a sliced run can be continued later
# without redoing finished units (state lives in /kaggle/working/results).
ONLY_FEEDERS = []
ONLY_KS = []

import os, sys, json, time, platform, warnings
warnings.filterwarnings("ignore")
t_start = time.time()

# Where the attached Kaggle dataset is mounted. Everything we need (code, .npz data and
# the precomputed design grids) lives under this one directory.
import glob as _g
_c = sorted(_g.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
DATA = os.path.dirname(_c[0]) if _c else "/kaggle/input/snapshot-fd-code-data/experiments"
print("DATA =", DATA)
if not os.path.isdir(DATA):
    # local fallback so the notebook also runs on a laptop for a smoke test
    DATA = os.environ.get("K4_DATA", DATA)
assert os.path.isdir(DATA), (
    "Attach the dataset 'snapshot-fd-code-data' (Add input -> Your datasets). "
    "Expected /kaggle/input/snapshot-fd-code-data/experiments")
# The dataset root also carries k4lib.py and e01.py next to experiments/.
# On Kaggle that is /kaggle/input/snapshot-fd-code-data; locally the task folder.
ROOT = os.environ.get("K4_ROOT", os.path.dirname(DATA))
for p in (DATA, ROOT):
    if p and p not in sys.path:
        sys.path.insert(0, p)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results")
os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS          # k4lib reads this
os.environ["K4_DATA"] = DATA

import numpy as np

# OpenDSS is not needed (design grids are precomputed); placement.py imports simlib,
# which imports opendssdirect at load time. Provide an empty stand-in if it is missing.
try:
    import opendssdirect  # noqa: F401
except ImportError:
    import types as _t
    sys.modules["opendssdirect"] = _t.ModuleType("opendssdirect")

import k4lib as K
import features as F
import placement as P
import placement2 as Q

# forward the optional work-slicing from the SETUP cell into the library
K.ONLY_FEEDERS = list(ONLY_FEEDERS)
K.ONLY_KS = list(ONLY_KS)

print("=== environment ===")
K.print_env()
print("SMOKE =", SMOKE)
if ONLY_FEEDERS:
    print("ONLY_FEEDERS =", ONLY_FEEDERS)
if ONLY_KS:
    print("ONLY_KS =", ONLY_KS)
print("elapsed", round(time.time() - t_start, 1), "s")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from lightgbm import LGBMClassifier

K_NN = 5            # neighbours per class for the margin
COV_GRID = np.round(np.linspace(0.05, 1.0, 20), 3)
TARGETS = (0.95, 0.99, 0.995)
REJECT_AT = 0.90    # coverage at which the share of HIF among rejected snapshots is reported


def noise_sd(ds, obs, noise, rep="FPS"):
    """Per-feature standard deviation of the measurement noise, from two independent draws."""
    X1 = K.features_of(ds, obs, 0, noise, rep)
    X2 = K.features_of(ds, obs, 1, noise, rep)
    sd = np.nanstd(X1 - X2, axis=0) / np.sqrt(2.0)
    return np.where(sd > 1e-12, sd, 1e-12)


def margin_scores(Xtr, ytr, Xte, classes):
    """Distance to the K_NN nearest training snapshots of each class (mean), whitened input.
    Returns (pred_by_margin, margin) for the test snapshots."""
    D = np.full((len(Xte), len(classes)), np.inf)
    for j, c in enumerate(classes):
        Xc = Xtr[ytr == c]
        if len(Xc) == 0:
            continue
        nn = NearestNeighbors(n_neighbors=min(K_NN, len(Xc))).fit(Xc)
        d, _ = nn.kneighbors(Xte)
        D[:, j] = d.mean(1)
    order = np.argsort(D, axis=1)
    first = D[np.arange(len(D)), order[:, 0]]
    second = D[np.arange(len(D)), order[:, 1]]
    return np.asarray(classes)[order[:, 0]], second - first


def risk_coverage(score, correct):
    """Accuracy of the retained fraction, keeping the highest scores first."""
    o = np.argsort(-score, kind="stable")
    c = correct[o].astype(float)
    cum = np.cumsum(c) / np.arange(1, len(c) + 1)
    return o, cum


def summarise(score, correct, rf):
    o, cum = risk_coverage(score, correct)
    n = len(o)
    out = {"aurc": float(np.mean(1 - cum))}
    for cov in COV_GRID:
        out[f"acc@{cov:.2f}"] = float(cum[max(1, int(round(cov * n))) - 1])
    for t in TARGETS:
        ok = np.where(cum >= t)[0]
        out[f"cov@{t}"] = float((ok[-1] + 1) / n) if len(ok) else 0.0   # LARGEST coverage
    n_keep = int(round(REJECT_AT * n))
    rej = o[n_keep:]
    out["hif_share_rejected"] = float(np.mean(rf[rej] > 10)) if len(rej) else float("nan")
    out["hif_share_all"] = float(np.mean(rf > 10))
    out["err_share_in_rejected"] = float((~correct[rej]).sum() / max(1, (~correct).sum()))
    return out

import json


In [ ]:
# ---------------- reject option: metrics per test fold, 2 seeds x 5 folds, k = 3
def fold_metrics(score, correct, rf):
    o = np.argsort(-score, kind="stable"); c = correct[o].astype(float)
    cum = np.cumsum(c) / np.arange(1, len(c) + 1); n = len(o)
    ok = np.where(cum >= 0.99)[0]
    cov99 = (ok[-1] + 1) / n if len(ok) else 0.0
    rej = o[int(round(0.9 * n)):]
    err = (~correct).sum()
    return dict(cov99=cov99,
                err_caught=((~correct[rej]).sum() / err) if err else np.nan,
                hif_rej=np.mean(rf[rej] > 10), hif_all=np.mean(rf > 10), acc=correct.mean())

rej_rows = []
for nb in K.feeder_list(SMOKE):
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    gi = K.rows_for_buses(ds, buses)
    _, marg, _ = P.margins(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
    obs = [gi[i] for i in P.dmp(marg, 3)]
    for noise in K.NOISE_MAIN:
        X = np.nan_to_num(K.features_of(ds, obs, 0, noise, "FPS"))
        Xw = X / noise_sd(ds, obs, noise)
        y, rf = ds.y, ds.rf; classes = np.unique(y)
        for seed in (0, 1):
            for fold, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y)):
                m = LGBMClassifier(**K.FAST).fit(X[tr], y[tr])
                pr = m.predict_proba(X[te]); pred = m.classes_[pr.argmax(1)]
                _, ms = margin_scores(Xw[tr], y[tr], Xw[te], classes)
                corr = pred == y[te]
                for name, sc in (("prob", pr.max(1)), ("margin", ms)):
                    r = fold_metrics(sc, corr, rf[te]); r.update(feeder=nb, noise=noise, score=name, seed=seed, fold=fold)
                    rej_rows.append(r)
        print("reject", nb, noise, "done", flush=True)
import pandas as pd
R = pd.DataFrame(rej_rows)
print(R.groupby(["feeder", "noise", "score"])[["acc", "cov99", "err_caught", "hif_rej", "hif_all"]].agg(["mean", "std"]).round(4).to_string())


In [ ]:
# ---------------- required accuracy: accuracy +- sd at each TVE class, k = 3
spec_rows = []
for nb in K.feeder_list(SMOKE):
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    gi = K.rows_for_buses(ds, buses)
    _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
    obs, _ = Q.dmp2(m2, wgt, 3)
    for tve in (0.005, 0.02, 0.1, 0.5, 1.0, 3.0):
        X = K.features_of(ds, obs, 0, K.tve_class(tve), "FPS")
        r = K.folds_acc(X, ds.y, ds.rf, seeds=(0, 1))
        spec_rows.append(dict(feeder=nb, tve_pct=tve, acc=r["acc"], acc_sd=r["acc_sd"], hif=r["hif"]))
        print(f"spec IEEE-{nb} TVE {tve}% acc {r['acc']*100:.1f} +- {r['acc_sd']*100:.1f}", flush=True)
    X = K.features_of(ds, obs, 0, K.NOISE["legacy 1%/0.5deg"], "FPS")
    r = K.folds_acc(X, ds.y, ds.rf, seeds=(0, 1))
    spec_rows.append(dict(feeder=nb, tve_pct="legacy", acc=r["acc"], acc_sd=r["acc_sd"], hif=r["hif"]))
    print(f"spec IEEE-{nb} legacy acc {r['acc']*100:.1f} +- {r['acc_sd']*100:.1f}", flush=True)


In [ ]:
# ---------------- comparison under the conditions of Khattak: same sensor sets as e12
E12 = {"ieee34": {"buses": {"Khattak FSNR-SVM": ["800", "848", "850", "854"], "DMP (proposed)": ["830", "848", "888", "890"], "DMP N-1 (proposed)": ["830", "854", "888", "890"], "Random 0": ["806", "860", "802", "818"], "Random 1": ["846", "806", "852", "860"], "Random 2": ["852", "814", "888", "862"], "Random 3": ["802", "854", "844", "856"], "Random 4": ["806", "858", "800", "854"]}}, "ieee123": {"buses": {"Khattak FSNR-SVM": ["149", "83", "151", "300", "95", "250", "66", "56", "450"], "DMP (proposed)": ["250", "49", "66", "91", "93", "95", "100", "105", "300_open"], "DMP N-1 (proposed)": ["30", "250", "151", "91", "93", "99", "450", "105", "300_open"], "Random 0": ["94_open", "102", "33", "43", "73", "24", "61s", "91", "67"], "Random 1": ["152", "21", "114", "85", "300", "2", "40", "160", "90"], "Random 2": ["36", "10", "62", "97", "100", "43", "135", "27", "32"], "Random 3": ["14", "23", "300", "15", "94_open", "102", "160", "105", "7"], "Random 4": ["151", "40", "114", "96", "56", "72", "8", "88", "51"]}}}   # sensor sets recorded by e12_khattak
CONDS = {"no noise, Rf<=10": ((0.01, 10.0), (0.0, 0.0)),
         "uPMU, all Rf": ((0, 1e9), K.NOISE["uPMU"]),
         "1% TVE, all Rf": ((0, 1e9), K.NOISE["1% TVE"])}
rep_rows = []
for tag, nb in (("ieee34", 34), ("ieee123", 123)):
    ds = K.dataset(nb); sets = E12[tag]["buses"]
    for cname, ((lo, hi), noise) in CONDS.items():
        keep = (ds.rf >= lo) & (ds.rf <= hi)
        for sname in ("Khattak FSNR-SVM", "DMP (proposed)") + tuple(f"Random {i}" for i in range(5)):
            obs = ds.bus_ids(sets[sname])
            X = K.features_of(ds, obs, 0, noise, "FPS")[keep]
            r = K.folds_acc(X, ds.y[keep], ds.rf[keep], seeds=(0, 1))
            rep_rows.append(dict(feeder=nb, cond=cname, set=sname, acc=r["acc"], acc_sd=r["acc_sd"]))
        print("repro", nb, cname, "done", flush=True)
Rp = pd.DataFrame(rep_rows)
Rp["group"] = Rp["set"].where(~Rp["set"].str.startswith("Random"), "Random (5 sets)")
print(Rp.groupby(["feeder", "cond", "group"])[["acc", "acc_sd"]].mean().round(4).to_string())


In [ ]:
R.to_csv(os.path.join(WORK, "S_reject_folds.csv"), index=False)
pd.DataFrame(spec_rows).to_csv(os.path.join(WORK, "S_spec_sd.csv"), index=False)
Rp.to_csv(os.path.join(WORK, "S_repro_sd.csv"), index=False)
print("TOTAL", round((time.time() - t_start) / 60, 1), "min")
